In [3]:
import os
import datetime
from openai import OpenAI
from elevenlabs.client import ElevenLabs
from dotenv import load_dotenv

In [8]:
# 1. Charger les clés API depuis .env
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
eleven_api_key = os.getenv("ELEVENLABS_API_KEY")

# 2. Définir la date du jour automatiquement
today = datetime.date.today()
today_str = today.strftime("%d %B %Y")  # exemple : "20 mai 2025"
filename = f"podcast_{today.strftime('%d/%m/%Y')}.mp3"

# 3. Initialiser OpenAI
client_openai = OpenAI(api_key=openai_api_key)

In [ ]:
# 4. Prompt dynamique avec la date
prompt = f"""
Tu es le créateur éditorial du podcast Simon FinTech, un podcast animé par un étudiant de 20 ans qui décrypte l’actualité de la finance et de la technologie dans un ton dynamique, accessible et bienveillant.
À partir de 4 à 5 articles d’actualité récents datés du {today_str}, rédige un script complet de podcast prêt à être lu directement par une IA vocale.
Tu dois uniquement t’appuyer sur des articles issus de sources fiables et bien référencées, comme Bloomberg, Reuters, Financial Times, WSJ, CNBC, TechCrunch, Les Échos, BFM Business, Wired, The Economist, etc. Tu ne dois jamais mentionner le nom de ces sources dans le texte.
Le script doit faire entre 1000 et 1200 mots pour une lecture à voix haute d’environ 7 à 8 minutes.
Chaque sujet abordé doit inclure : un contexte clair, une explication structurée des faits, et une mini-analyse ou projection personnelle.
Le ton doit être naturel, fluide et parlé, sans titres ni structure visible, avec des transitions naturelles entre les sujets, et sans aucune mention de sources ou liens.
Commence toujours par : "Bienvenue dans Simon FinTech, le podcast qui rend la finance et la tech simples, vivantes et surtout passionnantes."
Termine systématiquement par : "À demain pour un nouveau point sur l’actu tech !"
"""

# 5. Appel API GPT
response = client_openai.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7,
    max_tokens=2000
)

script_text = response.choices[0].message.content.strip()


In [1]:
print(script_text)

NameError: name 'script_text' is not defined

In [ ]:
# 6. Initialiser le client ElevenLabs
client_elevenlabs = ElevenLabs(api_key=eleven_api_key)

# 7. Génération audio avec ElevenLabs
audio = client_elevenlabs.text_to_speech.convert(
    text= "Hello World", #script_text,
    voice_id="pNInz6obpgDQGcFmaJgB",  # Remplace par l'ID de la voix souhaitée
    model_id="eleven_multilingual_v2",
    output_format="mp3_44100_128"
)

os.makedirs("audio", exist_ok=True)
with open(filename, "wb") as f:
    for chunk in audio:
        f.write(chunk)


print(f"✅ Podcast généré et sauvegardé : {filename}")


✅ Podcast généré et sauvegardé : podcast_20250521.mp3


In [12]:
from feedgen.feed import FeedGenerator
import datetime
import os

def generate_rss(audio_folder="podcasts", output_file="rss.xml"):
    fg = FeedGenerator()
    fg.load_extension('podcast')

    fg.title("Simon FinTech")
    fg.link(href="https://sarx613.github.io/Simon-FinTech/", rel="alternate")
    fg.logo("https://sarx613.github.io/Simon-FinTech/logo-podcast.png")
    fg.image("https://sarx613.github.io/Simon-FinTech/logo-podcast.png")
    fg.description("Le podcast qui rend la finance et la tech simples, vivantes et passionnantes.")
    fg.language("fr")
    fg.link(href="https://sarx613.github.io/Simon-FinTech/rss.xml", rel="self")

    for filename in sorted(os.listdir(audio_folder)):
        if filename.endswith(".mp3"):
            date_str = filename.replace("podcast_", "").replace(".mp3", "")
            date_obj = datetime.datetime.strptime(date_str, "%Y%m%d").replace(tzinfo=datetime.timezone.utc)

            episode = fg.add_entry()
            episode.id(f"https://sarx613.github.io/simon-fintech/podcasts/{filename}")
            episode.title(f"Simon FinTech – Actu du {date_obj.strftime('%d %B %Y')}")
            episode.description("Épisode quotidien de Simon FinTech.")
            episode.enclosure(
                url=f"https://sarx613.github.io/simon-fintech/podcasts/{filename}",
                length=0,
                type="audio/mpeg"
            )
            episode.pubDate(date_obj)

    fg.rss_file(output_file)
    print("✅ RSS mis à jour")

# Exemple d’utilisation
generate_rss()


✅ RSS mis à jour
